# Experiment: Observer construction costs on Gemma-2-9B-it

How much does each observer cost to construct and use? This notebook measures the existing dense residual, prompted, and Gemma Scope SAE observers on one A100. It is a cost study, not a new accuracy test.

Select an A100 runtime with high RAM and accept the Gemma-2-9B-it license on Hugging Face. The experiment reads APPS solutions as text; it never executes them.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/observerbench-costs')
REVISION = '80e23490d44630c90c151ad136528ff643072c5d'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/kwisatzh/observerbench.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', '--quiet', 'origin', REVISION], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '--quiet', '--detach', REVISION], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.14.1', 'accelerate==1.14.0', 'datasets==5.0.1', 'scikit-learn==1.6.1', 'pandas==2.2.3', 'numpy==2.1.3', '-e', str(ROOT)], check=True)
from huggingface_hub import login
login()  # Token stays out of notebook source and outputs.


## Plan

The original 500 fitting pairs and 200 calibration pairs determine the observers. A fixed 32-pair held-out panel measures warm inference cost. We retain the original context-limit rule.

We record model loading separately, then full-population measurements, layer/ridge selection and fitting, SAE encoding, and audit selection. Three warm repeats rotate method order. Sparse readout size is not treated as runtime. Downloads, model pretraining, and SAE pretraining are not included in warm inference.


In [ ]:
OUT = Path('/content/observerbench-cost-results')
runner = ROOT / 'scripts/measure_gemma_observer_costs.py'
assert runner.exists(), 'Use the repository revision containing the cost runner.'
subprocess.run([sys.executable, '-u', str(runner), '--outdir', str(OUT)], cwd=ROOT, check=True)


## Results

The summary adds the stages within each warm repeat before taking the median. The original A100 run measured 5.983 seconds for the neutral dense probe and 6.038 seconds for the SAE probe per 64 examples. Model inference dominated both. Construction, cold loading, and cached audit choice stay separate in timings.csv.


In [ ]:
import json, hashlib, shutil, tarfile
import pandas as pd
subprocess.run([sys.executable, str(ROOT / 'scripts/summarize_gemma_observer_costs.py'), str(OUT / 'results.json'), '--out', str(OUT / 'summary.json')], cwd=ROOT, check=True)
display(pd.DataFrame(json.loads((OUT / 'summary.json').read_text())['rows'])[['observer', 'median_seconds', 'min_seconds', 'max_seconds', 'max_gpu_allocated_gib']])
shutil.copy2(runner, OUT / runner.name)
archive = OUT.with_suffix('.tar.gz')
with tarfile.open(archive, 'w:gz') as bundle:
    bundle.add(OUT, arcname=OUT.name)
print('Archive SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))  # Preserve the complete run, not only its table.


## Next steps

Preserve the archive privately, and publish only the aggregate timing report and source fingerprints. Report the hardware, batch size, sequence lengths, and fitting population alongside every cost claim. These numbers do not establish a new monitor ranking or adaptive-attack robustness.
